In [1]:
import sys

print(sys.executable)
print(sys.version)

c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\venv\Scripts\python.exe
3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]


In [2]:
import pymupdf
import pandas as pd
import json

print("PyMuPDF: OK")
print("Pandas: OK")

PyMuPDF: OK
Pandas: OK


In [3]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

HLD_DIR = PROJECT_ROOT / "data" / "hld_documents"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("HLD directory:", HLD_DIR)
print("Processed directory:", PROCESSED_DIR)

Project root: c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant
HLD directory: c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\data\hld_documents
Processed directory: c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\data\processed


In [4]:
pdf_files = list(HLD_DIR.glob("*.pdf"))

print("PDF files found:", len(pdf_files))

for pdf in pdf_files:
    print("-", pdf.name)

PDF files found: 1
- sample_autosar_hld.pdf


In [5]:
# open hld file

import pymupdf

pdf_path = pdf_files[0]

doc = pymupdf.open(pdf_path)

print("Document:", pdf_path.name)
print("Total pages:", len(doc))

Document: sample_autosar_hld.pdf
Total pages: 5


In [6]:
page = doc[0]

text = page.get_text()

print("Page number:", 1)
print("Characters extracted:", len(text))
print()
print(text[:3000])

Page number: 1
Characters extracted: 941

Synthetic AUTOSAR HLD — Training/Demonstration Document
Page 1
AUTOSAR High-Level Design
EV Battery Monitoring and Thermal Management System
Document ID: EV-AUTOSAR-HLD-001
Version: 1.0
Status: Approved for demonstration
Purpose: Synthetic training document for the AUTOSAR HLD Document Analysis Assistant project.
1. System Overview
The system monitors battery voltage, battery temperature and state of charge. The BatteryMonitor software
component publishes battery status information. The ThermalController consumes battery temperature and
generates a cooling request. The Dashboard consumes battery status for driver display.
2. High-Level Architecture
The application layer contains BatteryMonitor, ThermalController and Dashboard. Communication between
application components is performed through defined ports and interfaces. The Runtime Environment (RTE)
provides the communication mechanism between application software components.



In [7]:
if len(text.strip()) == 0:
    print("⚠️ No text extracted. OCR may be required.")
else:
    print("✅ Text successfully extracted.")

✅ Text successfully extracted.


In [8]:
pages = []

for page_number, page in enumerate(doc, start=1):
    text = page.get_text().strip()

    page_record = {
        "document": pdf_path.name,
        "page": page_number,
        "text": text,
        "character_count": len(text),
        "needs_ocr_review": len(text) < 50
    }

    pages.append(page_record)

print("Total pages processed:", len(pages))

Total pages processed: 5


In [9]:
pages[0]

{'document': 'sample_autosar_hld.pdf',
 'page': 1,
 'text': 'Synthetic AUTOSAR HLD — Training/Demonstration Document\nPage 1\nAUTOSAR High-Level Design\nEV Battery Monitoring and Thermal Management System\nDocument ID: EV-AUTOSAR-HLD-001\nVersion: 1.0\nStatus: Approved for demonstration\nPurpose: Synthetic training document for the AUTOSAR HLD Document Analysis Assistant project.\n1. System Overview\nThe system monitors battery voltage, battery temperature and state of charge. The BatteryMonitor software\ncomponent publishes battery status information. The ThermalController consumes battery temperature and\ngenerates a cooling request. The Dashboard consumes battery status for driver display.\n2. High-Level Architecture\nThe application layer contains BatteryMonitor, ThermalController and Dashboard. Communication between\napplication components is performed through defined ports and interfaces. The Runtime Environment (RTE)\nprovides the communication mechanism between application soft

In [10]:
pages_df = pd.DataFrame(pages)

pages_df[["document", "page", "character_count", "needs_ocr_review"]]

,document,page,character_count,needs_ocr_review
0,sample_autosar_hld.pdf,1,940,False
1,sample_autosar_hld.pdf,2,836,False
2,sample_autosar_hld.pdf,3,829,False
3,sample_autosar_hld.pdf,4,927,False
4,sample_autosar_hld.pdf,5,1177,False


In [11]:
output_file = PROCESSED_DIR / "extracted_pages.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(pages, f, indent=4, ensure_ascii=False)

print("Saved successfully:")
print(output_file)

Saved successfully:
c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\data\processed\extracted_pages.json


In [12]:
for page in pages:
    print("\n" + "=" * 80)
    print(f"PAGE {page['page']}")
    print("=" * 80)
    print(page["text"])


PAGE 1
Synthetic AUTOSAR HLD — Training/Demonstration Document
Page 1
AUTOSAR High-Level Design
EV Battery Monitoring and Thermal Management System
Document ID: EV-AUTOSAR-HLD-001
Version: 1.0
Status: Approved for demonstration
Purpose: Synthetic training document for the AUTOSAR HLD Document Analysis Assistant project.
1. System Overview
The system monitors battery voltage, battery temperature and state of charge. The BatteryMonitor software
component publishes battery status information. The ThermalController consumes battery temperature and
generates a cooling request. The Dashboard consumes battery status for driver display.
2. High-Level Architecture
The application layer contains BatteryMonitor, ThermalController and Dashboard. Communication between
application components is performed through defined ports and interfaces. The Runtime Environment (RTE)
provides the communication mechanism between application software components.

PAGE 2
Synthetic AUTOSAR HLD — Training/Demonstrat

In [13]:
import re

heading_pattern = re.compile(
    r"^\d+(?:\.\d+)*\.\s+.+$"
)

detected_headings = []

for page in pages:
    lines = page["text"].splitlines()

    for line in lines:
        line = line.strip()

        if heading_pattern.match(line):
            detected_headings.append({
                "document": page["document"],
                "page": page["page"],
                "heading": line
            })

detected_headings

[{'document': 'sample_autosar_hld.pdf',
  'page': 1,
  'heading': '1. System Overview'},
 {'document': 'sample_autosar_hld.pdf',
  'page': 1,
  'heading': '2. High-Level Architecture'},
 {'document': 'sample_autosar_hld.pdf',
  'page': 2,
  'heading': '3. Software Components'},
 {'document': 'sample_autosar_hld.pdf',
  'page': 2,
  'heading': '4. Component Inventory'},
 {'document': 'sample_autosar_hld.pdf',
  'page': 3,
  'heading': '5. Ports and Interfaces'},
 {'document': 'sample_autosar_hld.pdf',
  'page': 3,
  'heading': '6. Interface Definitions'},
 {'document': 'sample_autosar_hld.pdf',
  'page': 4,
  'heading': '7. Signals and Data Flow'},
 {'document': 'sample_autosar_hld.pdf',
  'page': 4,
  'heading': '8. Functional Flow'},
 {'document': 'sample_autosar_hld.pdf',
  'page': 4,
  'heading': '1. BatteryMonitor reads battery voltage and temperature. 2. BatteryMonitor calculates BatteryStatus. 3.'},
 {'document': 'sample_autosar_hld.pdf',
  'page': 4,
  'heading': '9. Dependency 

In [14]:
# sections

In [15]:
import re

section_pattern = re.compile(
    r"^(\d+(?:\.\d+)*)\.\s+(.+)$"
)

sections = []
current_section = None

for page in pages:
    lines = page["text"].splitlines()

    for line in lines:
        line = line.strip()

        if not line:
            continue

        match = section_pattern.match(line)

        if match:
            if current_section is not None:
                sections.append(current_section)

            current_section = {
                "document": page["document"],
                "page": page["page"],
                "section_number": match.group(1),
                "section_title": match.group(2),
                "text": ""
            }

        elif current_section is not None:
            current_section["text"] += line + " "

if current_section is not None:
    sections.append(current_section)

print("Sections extracted:", len(sections))

Sections extracted: 14


In [16]:
sections

[{'document': 'sample_autosar_hld.pdf',
  'page': 1,
  'section_number': '1',
  'section_title': 'System Overview',
  'text': 'The system monitors battery voltage, battery temperature and state of charge. The BatteryMonitor software component publishes battery status information. The ThermalController consumes battery temperature and generates a cooling request. The Dashboard consumes battery status for driver display. '},
 {'document': 'sample_autosar_hld.pdf',
  'page': 1,
  'section_number': '2',
  'section_title': 'High-Level Architecture',
  'text': 'The application layer contains BatteryMonitor, ThermalController and Dashboard. Communication between application components is performed through defined ports and interfaces. The Runtime Environment (RTE) provides the communication mechanism between application software components. Synthetic AUTOSAR HLD — Training/Demonstration Document Page 2 '},
 {'document': 'sample_autosar_hld.pdf',
  'page': 2,
  'section_number': '3',
  'sectio

In [17]:
sections_df = pd.DataFrame(sections)

sections_df[
    [
        "page",
        "section_number",
        "section_title"
    ]
]

,page,section_number,section_title
0,1,1,System Overview
1,1,2,High-Level Architecture
2,2,3,Software Components
3,2,4,Component Inventory
4,3,5,Ports and Interfaces
5,3,6,Interface Definitions
6,4,7,Signals and Data Flow
7,4,8,Functional Flow
8,4,1,BatteryMonitor reads battery voltage and tempe...
9,4,9,Dependency Mapping


In [18]:
section_output = PROCESSED_DIR / "extracted_sections.json"

with open(section_output, "w", encoding="utf-8") as f:
    json.dump(sections, f, indent=4, ensure_ascii=False)

print("Sections saved to:")
print(section_output)

Sections saved to:
c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\data\processed\extracted_sections.json


In [19]:
for section in sections:
    print(
        f"{section['section_number']}. "
        f"{section['section_title']} "
        f"(Page {section['page']})"
    )

1. System Overview (Page 1)
2. High-Level Architecture (Page 1)
3. Software Components (Page 2)
4. Component Inventory (Page 2)
5. Ports and Interfaces (Page 3)
6. Interface Definitions (Page 3)
7. Signals and Data Flow (Page 4)
8. Functional Flow (Page 4)
1. BatteryMonitor reads battery voltage and temperature. 2. BatteryMonitor calculates BatteryStatus. 3. (Page 4)
9. Dependency Mapping (Page 4)
10. Architecture Relationships (Page 5)
11. Validation Rules (Page 5)
12. Revision Information (Page 5)
13. Example Questions for the AI Assistant (Page 5)


In [20]:
import re

heading_pattern = re.compile(
    r"^(\d+(?:\.\d+)*)\.\s+(.+)$"
)

sections_v2 = []
current_section = None

for page in pages:
    lines = page["text"].splitlines()

    for line in lines:
        line = line.strip()

        # Remove PDF header/footer
        if line == "Synthetic AUTOSAR HLD — Training/Demonstration Document":
            continue

        if re.fullmatch(r"Page \d+", line):
            continue

        if not line:
            continue

        match = heading_pattern.match(line)

        if match:
            section_number = match.group(1)
            section_title = match.group(2).strip()

            # Only treat short numbered lines as headings.
            # Sentence-like numbered lines are treated as normal content.
            is_likely_heading = (
                len(section_title) <= 60
                and not section_title.endswith(".")
            )

            if is_likely_heading:
                if current_section is not None:
                    sections_v2.append(current_section)

                current_section = {
                    "document": page["document"],
                    "page": page["page"],
                    "section_number": section_number,
                    "section_title": section_title,
                    "text": ""
                }

                continue

        if current_section is not None:
            current_section["text"] += line + " "

if current_section is not None:
    sections_v2.append(current_section)

print("Sections extracted:", len(sections_v2))

Sections extracted: 13


In [21]:
for section in sections_v2:
    print(
        f"{section['section_number']} "
        f"{section['section_title']} "
        f"(Page {section['page']})"
    )

1 System Overview (Page 1)
2 High-Level Architecture (Page 1)
3 Software Components (Page 2)
4 Component Inventory (Page 2)
5 Ports and Interfaces (Page 3)
6 Interface Definitions (Page 3)
7 Signals and Data Flow (Page 4)
8 Functional Flow (Page 4)
9 Dependency Mapping (Page 4)
10 Architecture Relationships (Page 5)
11 Validation Rules (Page 5)
12 Revision Information (Page 5)
13 Example Questions for the AI Assistant (Page 5)


In [22]:
for section in sections_v2:
    section["level"] = section["section_number"].count(".") + 1

In [23]:
sections_v2_df = pd.DataFrame(sections_v2)

sections_v2_df[
    [
        "page",
        "section_number",
        "section_title",
        "level"
    ]
]

,page,section_number,section_title,level
0,1,1,System Overview,1
1,1,2,High-Level Architecture,1
2,2,3,Software Components,1
3,2,4,Component Inventory,1
4,3,5,Ports and Interfaces,1
5,3,6,Interface Definitions,1
6,4,7,Signals and Data Flow,1
7,4,8,Functional Flow,1
8,4,9,Dependency Mapping,1
9,5,10,Architecture Relationships,1


In [24]:
improved_section_output = PROCESSED_DIR / "extracted_sections_v2.json"

with open(improved_section_output, "w", encoding="utf-8") as f:
    json.dump(sections_v2, f, indent=4, ensure_ascii=False)

print("Saved:")
print(improved_section_output)

Saved:
c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\data\processed\extracted_sections_v2.json
